# Módulo de Configuración Histórica
## Notebook 00 — Tablas BSC Dimensiones

Este notebook crea y carga las cuatro tablas de dimensiones con prefijo `bsc_`:

| Tabla | Descripción |
|---|---|
| `bsc_dim_usuario` | Personas registradas en el sistema |
| `bsc_dim_rol` | Roles o posiciones dentro del modelo comercial |
| `bsc_dim_evaluacion` | Indicadores o métricas que se miden |
| `bsc_dim_esquema` | Esquemas de compensación disponibles |

> **Prerrequisito**: Asegúrate de que este notebook esté adjunto a un Lakehouse antes de ejecutarlo.  
> En la barra lateral de Fabric selecciona **Add Lakehouse** y elige tu Lakehouse de destino.

---
## 0 · Configuración del Lakehouse

In [ ]:
# ─── Ajusta este valor al nombre de tu Lakehouse en Fabric ───────────────────
LAKEHOUSE_NAME = "BI - Bandelta LH"
# ─────────────────────────────────────────────────────────────────────────────

spark.sql(f"USE `{LAKEHOUSE_NAME}`")
print(f"✅ Usando Lakehouse: {LAKEHOUSE_NAME}")

---
## 1 · bsc_dim_usuario

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bsc_dim_usuario (
    id_usuario      INT           NOT NULL  COMMENT 'Llave primaria del usuario',
    nombre          STRING        NOT NULL  COMMENT 'Nombre completo',
    email           STRING        NOT NULL  COMMENT 'Correo electrónico corporativo (único)',
    codigo_empleado STRING                  COMMENT 'Número de empleado en el sistema RH',
    area            STRING                  COMMENT 'Área o departamento al que pertenece',
    activo          BOOLEAN       NOT NULL  COMMENT 'Indica si el usuario está activo',
    fecha_creacion  TIMESTAMP     NOT NULL  COMMENT 'Timestamp de alta en el sistema',
    fecha_baja      TIMESTAMP               COMMENT 'Timestamp de baja; NULL si sigue activo'
)
USING DELTA
COMMENT 'Catálogo de usuarios del módulo de compensación histórica'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla bsc_dim_usuario creada (o ya existía).")

In [ ]:
# ─── 1. Define los registros a cargar ────────────────────────────────────────
datos_usuario = [
    # Descomenta o agrega filas según necesites
    {"id_usuario": 1, "nombre": "Nombre Ejemplo", "email": "usuario@empresa.com",
     "codigo_empleado": "EMP-001", "area": "Ventas", "activo": True,
     "fecha_creacion": "2024-01-01T00:00:00", "fecha_baja": None},
    # {"id_usuario": 2, "nombre": "...", "email": "...",
    #  "codigo_empleado": "...", "area": "...", "activo": True,
    #  "fecha_creacion": "...", "fecha_baja": None},
]

# ─── 2. Validación previa ─────────────────────────────────────────────────────
from pyspark.sql import functions as F

df_u = spark.createDataFrame(datos_usuario)

# 2a. Duplicados dentro del batch
dup = df_u.groupBy("id_usuario").count().filter("count > 1")
assert dup.count() == 0, f"❌ IDs duplicados en el batch: {dup.collect()}"

# 2b. Nulos en columnas NOT NULL
for col_nn in ["id_usuario", "nombre", "email", "activo", "fecha_creacion"]:
    nulls = df_u.filter(F.col(col_nn).isNull()).count()
    assert nulls == 0, f"❌ La columna '{col_nn}' contiene {nulls} valor(es) nulo(s)"

# 2c. Emails únicos dentro del batch
dup_email = df_u.groupBy("email").count().filter("count > 1")
assert dup_email.count() == 0, f"❌ Emails duplicados en el batch: {dup_email.collect()}"

print("✅ Validación pasada — procediendo con MERGE")

# ─── 3. MERGE (upsert idempotente) ───────────────────────────────────────────
df_u.createOrReplaceTempView("_stage_usuario")

spark.sql("""
MERGE INTO bsc_dim_usuario AS target
USING _stage_usuario       AS source
ON target.id_usuario = source.id_usuario
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *
""")

spark.sql("SELECT * FROM bsc_dim_usuario ORDER BY id_usuario").show(truncate=False)

---
## 2 · bsc_dim_rol

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bsc_dim_rol (
    id_rol       INT     NOT NULL  COMMENT 'Llave primaria del rol',
    nombre_rol   STRING  NOT NULL  COMMENT 'Nombre del rol, p.ej. Asesor Comercial',
    descripcion  STRING            COMMENT 'Descripción detallada del rol',
    nivel        STRING            COMMENT 'Nivel jerárquico: Operativo, Táctico, Estratégico',
    activo       BOOLEAN NOT NULL  COMMENT 'Indica si el rol está vigente'
)
USING DELTA
COMMENT 'Catálogo de roles del modelo de compensación'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla bsc_dim_rol creada (o ya existía).")

In [ ]:
# ─── 1. Define los registros a cargar ────────────────────────────────────────
datos_rol = [
    {"id_rol": 1, "nombre_rol": "Asesor Comercial",
     "descripcion": "Venta directa a cliente final", "nivel": "Operativo", "activo": True},
    # {"id_rol": 2, "nombre_rol": "...", "descripcion": "...", "nivel": "...", "activo": True},
]

# ─── 2. Validación previa ─────────────────────────────────────────────────────
df_r = spark.createDataFrame(datos_rol)

# 2a. Duplicados dentro del batch
dup = df_r.groupBy("id_rol").count().filter("count > 1")
assert dup.count() == 0, f"❌ IDs duplicados en el batch: {dup.collect()}"

# 2b. Nulos en columnas NOT NULL
for col_nn in ["id_rol", "nombre_rol", "activo"]:
    nulls = df_r.filter(F.col(col_nn).isNull()).count()
    assert nulls == 0, f"❌ La columna '{col_nn}' contiene {nulls} valor(es) nulo(s)"

# 2c. Nivel válido
niveles_validos = {"Operativo", "Táctico", "Estratégico"}
niveles_batch = {r["nivel"] for r in datos_rol if r.get("nivel")}
invalidos = niveles_batch - niveles_validos
assert not invalidos, f"❌ Niveles no permitidos: {invalidos}. Permitidos: {niveles_validos}"

print("✅ Validación pasada — procediendo con MERGE")

# ─── 3. MERGE (upsert idempotente) ───────────────────────────────────────────
df_r.createOrReplaceTempView("_stage_rol")

spark.sql("""
MERGE INTO bsc_dim_rol AS target
USING _stage_rol       AS source
ON target.id_rol = source.id_rol
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *
""")

spark.sql("SELECT * FROM bsc_dim_rol ORDER BY id_rol").show(truncate=False)

---
## 3 · bsc_dim_evaluacion

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bsc_dim_evaluacion (
    id_evaluacion      INT     NOT NULL  COMMENT 'Llave primaria de la evaluación',
    nombre_evaluacion  STRING  NOT NULL  COMMENT 'Nombre del indicador, p.ej. Ventas Netas',
    descripcion        STRING            COMMENT 'Qué mide y cómo se calcula',
    tipo_metrica       STRING  NOT NULL  COMMENT 'porcentaje | valor_absoluto | ratio | conteo',
    unidad_medida      STRING            COMMENT 'MXN, unidades, %, clientes, etc.',
    activo             BOOLEAN NOT NULL  COMMENT 'Indica si la evaluación está vigente'
)
USING DELTA
COMMENT 'Catálogo de evaluaciones/indicadores del modelo de compensación'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla bsc_dim_evaluacion creada (o ya existía).")

In [ ]:
# ─── 1. Define los registros a cargar ────────────────────────────────────────
datos_evaluacion = [
    {"id_evaluacion": 1, "nombre_evaluacion": "Ventas Netas",
     "descripcion": "Monto total de ventas netas", "tipo_metrica": "valor_absoluto",
     "unidad_medida": "MXN", "activo": True},
    # {"id_evaluacion": 2, "nombre_evaluacion": "...", "descripcion": "...",
    #  "tipo_metrica": "...", "unidad_medida": "...", "activo": True},
]

# ─── 2. Validación previa ─────────────────────────────────────────────────────
df_e = spark.createDataFrame(datos_evaluacion)

# 2a. Duplicados dentro del batch
dup = df_e.groupBy("id_evaluacion").count().filter("count > 1")
assert dup.count() == 0, f"❌ IDs duplicados en el batch: {dup.collect()}"

# 2b. Nulos en columnas NOT NULL
for col_nn in ["id_evaluacion", "nombre_evaluacion", "tipo_metrica", "activo"]:
    nulls = df_e.filter(F.col(col_nn).isNull()).count()
    assert nulls == 0, f"❌ La columna '{col_nn}' contiene {nulls} valor(es) nulo(s)"

# 2c. tipo_metrica válido
tipos_validos = {"porcentaje", "valor_absoluto", "ratio", "conteo"}
tipos_batch = {r["tipo_metrica"] for r in datos_evaluacion}
invalidos = tipos_batch - tipos_validos
assert not invalidos, f"❌ tipo_metrica no permitido: {invalidos}. Permitidos: {tipos_validos}"

print("✅ Validación pasada — procediendo con MERGE")

# ─── 3. MERGE (upsert idempotente) ───────────────────────────────────────────
df_e.createOrReplaceTempView("_stage_evaluacion")

spark.sql("""
MERGE INTO bsc_dim_evaluacion AS target
USING _stage_evaluacion       AS source
ON target.id_evaluacion = source.id_evaluacion
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *
""")

spark.sql("SELECT * FROM bsc_dim_evaluacion ORDER BY id_evaluacion").show(truncate=False)

---
## 4 · bsc_dim_esquema

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bsc_dim_esquema (
    id_esquema      INT     NOT NULL  COMMENT 'Llave primaria del esquema',
    nombre_esquema  STRING  NOT NULL  COMMENT 'Nombre descriptivo, p.ej. Esquema Ventas 2024',
    descripcion     STRING            COMMENT 'Objetivo y alcance del esquema',
    tipo_esquema    STRING            COMMENT 'individual | grupal | mixto',
    activo          BOOLEAN NOT NULL  COMMENT 'Indica si el esquema está vigente'
)
USING DELTA
COMMENT 'Catálogo de esquemas de compensación'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla bsc_dim_esquema creada (o ya existía).")

In [ ]:
# ─── 1. Define los registros a cargar ────────────────────────────────────────
datos_esquema = [
    {"id_esquema": 1, "nombre_esquema": "Esquema Ventas 2024",
     "descripcion": "Compensación por volumen de ventas", "tipo_esquema": "individual", "activo": True},
    # {"id_esquema": 2, "nombre_esquema": "...", "descripcion": "...",
    #  "tipo_esquema": "...", "activo": True},
]

# ─── 2. Validación previa ─────────────────────────────────────────────────────
df_es = spark.createDataFrame(datos_esquema)

# 2a. Duplicados dentro del batch
dup = df_es.groupBy("id_esquema").count().filter("count > 1")
assert dup.count() == 0, f"❌ IDs duplicados en el batch: {dup.collect()}"

# 2b. Nulos en columnas NOT NULL
for col_nn in ["id_esquema", "nombre_esquema", "activo"]:
    nulls = df_es.filter(F.col(col_nn).isNull()).count()
    assert nulls == 0, f"❌ La columna '{col_nn}' contiene {nulls} valor(es) nulo(s)"

# 2c. tipo_esquema válido
tipos_validos = {"individual", "grupal", "mixto"}
tipos_batch = {r["tipo_esquema"] for r in datos_esquema if r.get("tipo_esquema")}
invalidos = tipos_batch - tipos_validos
assert not invalidos, f"❌ tipo_esquema no permitido: {invalidos}. Permitidos: {tipos_validos}"

print("✅ Validación pasada — procediendo con MERGE")

# ─── 3. MERGE (upsert idempotente) ───────────────────────────────────────────
df_es.createOrReplaceTempView("_stage_esquema")

spark.sql("""
MERGE INTO bsc_dim_esquema AS target
USING _stage_esquema       AS source
ON target.id_esquema = source.id_esquema
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *
""")

spark.sql("SELECT * FROM bsc_dim_esquema ORDER BY id_esquema").show(truncate=False)

---
## 5 · Resumen de tablas

In [ ]:
tablas_bsc = ["bsc_dim_usuario", "bsc_dim_rol", "bsc_dim_evaluacion", "bsc_dim_esquema"]

print(f"{'TABLA':<25} {'REGISTROS':>10}")
print("-" * 37)
for tabla in tablas_bsc:
    cnt = spark.sql(f"SELECT COUNT(*) as c FROM {tabla}").collect()[0].c
    print(f"{tabla:<25} {cnt:>10,}")

print("\n✅ Notebook 00 completado. Continúa con 01_dimensiones_config_historico.ipynb")